### Extraction des annotations Doccano vers un CSV tabulaire.

Ce script lit un fichier JSONL exporté depuis Doccano, récupère les entités
médicamenteuses et leurs attributs via les relations annotées, puis exporte
un fichier CSV exploitable pour l'analyse ou la comparaison avec Medkit.


In [1]:
import json
import pandas as pd
import itertools
from collections import defaultdict

# Fichiers d'entrée / sortie
INPUT_JSONL = "doccanno0904.jsonl"
OUTPUT_CSV = "gold_standard.csv"

# Labels utilisés dans le projet
MED_LABELS = {"Med", "Classe"}
DOSAGE_LABELS = {"Dosage"}
FREQ_LABELS = {"Freq"}
ROUTE_LABELS = {"Route"}
DATE_LABELS = {"Date"}
DATE_REL_LABELS = {"Date_relative", "Duree"}
COND_LABELS = {"Contexte"}

# Relations que l'on garde pour reconstruire les liens
REL_LINK = {"Refer_to", "Start", "Stop", "Augmentation", "Duration_presc"}


def join_unique(values):
    "Concatène des valeurs en supprimant les doublons tout en gardant l'ordre"
    out = []
    for v in values:
        v = (v or "").strip()
        if v and v not in out:
            out.append(v)
    return " | ".join(out)

In [2]:
# Lecture du JSONL Doccano
rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        text = item.get("text", "")
        ents = item.get("entities", [])
        rels = item.get("relations", [])

        # Reconstruction des entités avec leurs offsets dans le texte
        ents_by_id = {}
        for e in ents:
            eid = int(e["id"])
            sid = int(e["start_offset"])
            end = int(e["end_offset"])
            ents_by_id[eid] = {
                "id": eid,
                "label": str(e["label"]).strip(),
                "start": sid,
                "end": end,
                "text": text[sid:end].strip(),
            }

        # Médicaments / classes repérés dans le document
        meds = sorted(
            [e for e in ents_by_id.values() if e["label"] in MED_LABELS],
            key=lambda x: x["start"]
        )

        # Association médicament -> attributs via les relations
        med_links = defaultdict(list)

        for r in rels:
            rtype = str(r.get("type", "")).strip()
            if rtype not in REL_LINK:
                continue

            try:
                from_id = int(r.get("from_id"))
                to_id = int(r.get("to_id"))
            except (TypeError, ValueError):
                continue

            if from_id not in ents_by_id or to_id not in ents_by_id:
                continue

            src = ents_by_id[from_id]
            tgt = ents_by_id[to_id]

            # Si la cible est le médicament et la source est un attribut
            if tgt["label"] in MED_LABELS and src["label"] not in MED_LABELS:
                med_links[to_id].append(from_id)

            # Si la source est le médicament et la cible est un attribut
            elif src["label"] in MED_LABELS and tgt["label"] not in MED_LABELS:
                med_links[from_id].append(to_id)

        # Construction des lignes du tableau final
        for med in meds:
            mid = med["id"]
            linked = sorted(
                [ents_by_id[x] for x in med_links.get(mid, []) if x in ents_by_id],
                key=lambda x: x["start"]
            )

            def get_vals(label_set):
                vals = list(dict.fromkeys(
                    e["text"] for e in linked if e["label"] in label_set
                ))
                return vals if vals else [""]

            dosages = get_vals(DOSAGE_LABELS)
            frequences = get_vals(FREQ_LABELS)
            voies = get_vals(ROUTE_LABELS)
            dates = get_vals(DATE_LABELS)
            dates_rel = get_vals(DATE_REL_LABELS)
            contextes = get_vals(COND_LABELS)

            # Produit cartésien pour garder une ligne par combinaison d'attributs
            seen = set()
            for (d, f, v, dt, dr, c) in itertools.product(
                dosages, frequences, voies, dates, dates_rel, contextes
            ):
                key = (med["text"], med["start"], d, f, v, dt, dr, c)
                if key in seen:
                    continue
                seen.add(key)

                rows.append({
                    "Medicament": med["text"],
                    "Label_type": med["label"],
                    "Dosage": d,
                    "Frequence": f,
                    "Voie_administration": v,
                    "Date": dt,
                    "Date_relative": dr,
                    "Contexte": c,
                    "Position_texte": med["start"],
                })

# Export CSV
df = pd.DataFrame(rows, columns=[
    "Medicament", "Label_type", "Dosage", "Frequence",
    "Voie_administration", "Date", "Date_relative",
    "Contexte", "Position_texte"
])

df.to_csv(OUTPUT_CSV, index=True, index_label="ID", encoding="utf-8-sig")
print(f"{OUTPUT_CSV} — lignes: {len(df)}")

gold_standard.csv — lignes: 129
